# Day 6 · Pandas 进阶(2/2)

目标:groupby 分组聚合、merge 表合并、transform 组内计算。

今日节奏:40min 学习 + 15min 动手 + 5min 自检(最后有清单)。

> 打开方式:JupyterLab 文件树里进 `ai-learning/练习/` 双击本文件,逐格 Shift+Enter。


In [ ]:
import sys
import numpy as np
import pandas as pd

print("Python", sys.version.split()[0], "| pandas", pd.__version__)
print("环境 OK!开始今天的练习 →")


## 任务 1:groupby 三阶段(split-apply-combine)

`df.groupby("列名")` 把数据按取值**分组**,对每组**计算**,再**合并**成一个结果表。

- `groupby("部门")["收入"].sum()` → 每组求和
- `.agg(["mean", "max", "count"])` → 一个列同时算多个统计量
- `.agg({"收入": "sum", "成本": "mean"})` → 不同列不同统计量


In [ ]:
sales = pd.DataFrame({
    "部门": ["技术", "销售", "销售", "技术", "销售", "技术"],
    "季度": ["Q1", "Q1", "Q2", "Q2", "Q1", "Q2"],
    "收入": [120, 200, 180, 150, 220, 160],
    "成本": [80, 130, 120, 95, 140, 100],
})
print("按部门收入求和:\n", sales.groupby("部门")["收入"].sum())
print("多统计量:\n", sales.groupby("部门")["收入"].agg(["mean", "max", "count"]))
print("多列不同统计:\n", sales.groupby("部门").agg({"收入": "sum", "成本": "mean"}).round(1))


## 任务 2:多列分组与透视表

- `groupby(["部门", "季度"])` → 两层分组;`.reset_index()` 把结果变回普通表格
- `pd.pivot_table(...)` → 交叉表:行=部门、列=季度、格子=收入合计


In [ ]:
both = sales.groupby(["部门", "季度"])["收入"].sum().reset_index()
print("按部门+季度:\n", both)

piv = pd.pivot_table(
    sales,
    index="部门",
    columns="季度",
    values="收入",
    aggfunc="sum",
    fill_value=0,
)
print("透视表:\n", piv)


## 任务 3:merge 合并两张表

数据库里的 JOIN,pandas 里叫 merge。`how` 决定保留哪边:

- `inner` 两边都有的键才保留
- `left` 左表全保留,右表对不上就补 NaN
- `right` / `outer` 同理(右全留 / 两边全留)


In [ ]:
customers = pd.DataFrame({
    "客户ID": [1, 2, 3, 4],
    "城市": ["北京", "上海", "广州", "深圳"],
})
orders = pd.DataFrame({
    "订单ID": [101, 102, 103, 104, 105],
    "客户ID": [1, 2, 2, 3, 9],
    "金额": [300, 500, 200, 800, 999],
})
print("inner(两边都有):\n", orders.merge(customers, on="客户ID", how="inner"))

left = orders.merge(customers, on="客户ID", how="left").fillna({"城市": "未知"})
print("left(订单全保留):\n", left)


## 任务 4:transform 组内计算

`agg` 把一组压成**一个数**,`transform` 给组内**每一行**一个结果(长度和原表一样)。

典型用法:算"组内平均",再算每行偏离组平均多少——这就是以后做组内标准化的套路。


In [ ]:
sales["组内平均"] = sales.groupby("部门")["收入"].transform("mean")
sales["偏离组平均"] = sales["收入"] - sales["组内平均"]
print(sales)


## 任务 5:小挑战 🔥(merge + groupby 连招)

把订单和客户信息合并,然后回答:每个城市的总金额、订单数和平均单价各是多少?哪个城市消费最高?


In [ ]:
merged = orders.merge(customers, on="客户ID", how="left").fillna({"城市": "未知"})
city = merged.groupby("城市").agg(
    订单数=("订单ID", "count"),
    总额=("金额", "sum"),
    单均=("金额", "mean"),
).round(1)
print(city.sort_values("总额", ascending=False))


## 自检清单(5 问,答不上就回看今天的格子)

1. groupby 的三阶段? → split 分组 / apply 计算 / combine 合并
2. `.agg({"A": "sum", "B": "mean"})` 是什么? → 列 A 求和、列 B 求平均
3. `how="left"` 的 merge 保留谁? → 左表全保留,右表配不上的位置补 NaN
4. agg 和 transform 返回形状的区别? → agg 每组一个数(变短);transform 与原表等长
5. 什么时候用 pivot_table? → 想要"行×列"的交叉汇总时

## 📝 收盘动作

```powershell
cd D:\01_Study\ai-learning
git add -A; git commit -m "day6: pandas advanced"; git push
```

然后跟助手说"生成日志"。
